# API参考: `SimulationController`

**类路径:** `common.controller.SimulationController`

## 目的
`SimulationController` 是整个耦合模型框架的“指挥中心”。它负责：
- 容纳所有的模型组件（如`HydrologicalModel`, `HydraulicModel`, `Junction`等）。
- 管理这些组件之间的连接关系，形成一个计算网络。
- 确定正确的计算顺序（通过拓扑排序）。
- 处理复杂的网络拓扑（如环路）的求解。
- 在每个时间步，协调数据在组件之间的传递，并调用每个组件的`step`方法。

## `__init__(self)`

构造函数。创建一个空的控制器，包含空的组件字典、网络定义等。

### 示例

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from common.controller import SimulationController

controller = SimulationController()
print("空的SimulationController创建成功。")

## `add_component(self, component)`

将一个模型组件实例添加到控制器中。

**参数:**
- `component` (BaseModelComponent): 一个继承自`BaseModelComponent`的组件实例。

### 示例

In [ ]:
from common.junction import Junction

junction_comp = Junction(name="J1")
controller.add_component(junction_comp)

print(f"成功添加组件: {list(controller.components.keys())}")

## `connect(self, upstream_name, downstream_name)`

定义两个已添加的组件之间的连接关系（水流方向）。

**参数:**
- `upstream_name` (str): 上游组件的名称（`name`属性）。
- `downstream_name` (str): 下游组件的名称。

### 示例

In [ ]:
from common.base_model import BaseModelComponent

# 创建两个虚拟的源和汇组件用于演示
class Source(BaseModelComponent):
    def step(self, inflows, dt): self.outflow = 10
class Sink(BaseModelComponent):
    def step(self, inflows, dt): self.outflow = sum(inflows.values())
        
source1 = Source(name="Upstream1")
sink1 = Sink(name="Downstream1")
controller.add_component(source1)
controller.add_component(sink1)

controller.connect("Upstream1", "Downstream1")
print("成功连接 Upstream1 -> Downstream1")
print(f"网络定义: {controller.network}")

## `run(self, num_steps, dt, global_inputs=None, log_history=False)`

启动并执行整个网络模拟。

**参数:**
- `num_steps` (int): 模拟的总时间步数。
- `dt` (float): 每个时间步的长度（秒）。
- `global_inputs` (dict, optional): 一个可选的字典，用于提供全局输入（如降雨）。其键是组件名称，值是另一个包含输入名称和时间序列的字典。
- `log_history` (bool, optional): 如果设为`True`，`run`方法将返回一个列表，其中包含了每个时间步所有组件的详细状态历史。默认为`False`，此时`run`是一个生成器，只在每步结束后`yield`一个简单的状态。

### 示例

In [ ]:
# 使用上面定义的包含Source和Sink的网络
print("运行一个简单的网络模拟...")
history = controller.run(num_steps=3, dt=1, log_history=True)

print("\n模拟完成。查看最后一个时间步的历史记录:")
import json
print(json.dumps(history[-1], indent=2))